# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [5]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
#Function to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    # api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [7]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['employees', 'salary', 'studies'],
        'definition': ['Employee information, name...',
                       'Salary details for each year',
                       'Educational studies, name of the institution, type of studies, level']}
df = pd.DataFrame(data)
print(df)

       table                                         definition
0  employees                      Employee information, name...
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [14]:
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level


In [16]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [18]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question=#ENTER YOUR QUERY HERE)

In [24]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "salary"]
}
```


In [26]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question=#ENTER YOUR QUERY HERE)

In [28]:
print(return_OAI(pqt3))

```json
{
    "tables": {
        "employees": "Employee information",
        "salary": "Salary details for each year",
        "studies": "Educational studies"
    }
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [9]:
import pandas as pd

data = {
    "table": [
        "employees",
        "salary",
        "studies"
    ],
    "definition": [
        "Contains employee personal information such as id, name, department.",
        "Contains employee salary per year including employee_id, year, amount.",
        "Contains education background including employee_id, degree, field_of_study."
    ]
}

df = pd.DataFrame(data)
df

,table,definition
0,employees,Contains employee personal information such as...
1,salary,Contains employee salary per year including em...
2,studies,Contains education background including employ...


In [10]:
tables_text = ""

for i, row in df.iterrows():
    tables_text += f"{row['table']}: {row['definition']}\n"

print(tables_text)

employees: Contains employee personal information such as id, name, department.
salary: Contains employee salary per year including employee_id, year, amount.
studies: Contains education background including employee_id, degree, field_of_study.



In [11]:
questions = [
    "Who is the highest paid employee?",
    "List employees and their field of study.",
    "What is the average salary in 2022?",
    "Which employees studied engineering and earn more than 50000?"
]

In [17]:
def prompt_v1(question):
    return f"""
Given the following tables and their definitions:

{tables_text}

Tell me which tables are necessary to answer the question.
Return the table names.

Question:
{question}
"""

In [12]:
def prompt_v2(question):
    return f"""
You are a database assistant.

Only use the tables provided below.
Do NOT invent tables.
Return ONLY valid JSON in this format:
{{"tables": ["table1", "table2"]}}

Available tables:
{tables_text}

Question:
{question}
"""

In [13]:
def prompt_v3(question):
    return f"""
You are an expert SQL planner.

Task:
Select ONLY the minimum necessary tables required to answer the question.

Rules:
- Only use tables listed below.
- Do not explain.
- Do not invent tables.
- Output strictly valid JSON:
{{"tables": ["table1", "table2"]}}

Available tables:
{tables_text}

Question:
{question}
"""

In [14]:
def prompt_v4(question):
    return f"""
You are a SQL table selection system.

Example:
Tables:
employees: employee information
salary: salary per employee
studies: education background

Question:
Who is the highest paid employee?

Answer:
{{"tables": ["employees", "salary"]}}

Now solve:

Tables:
{tables_text}

Question:
{question}

Answer:
"""

In [15]:
def test_all_prompts():
    for i, q in enumerate(questions):
        print("====================================")
        print(f"QUESTION {i+1}: {q}")
        print("====================================\n")
        
        print("---- Prompt V1 ----")
        print(return_OAI(prompt_v1(q)))
        print()
        
        print("---- Prompt V2 ----")
        print(return_OAI(prompt_v2(q)))
        print()
        
        print("---- Prompt V3 ----")
        print(return_OAI(prompt_v3(q)))
        print()
        
        print("---- Prompt V4 ----")
        print(return_OAI(prompt_v4(q)))
        print("\n\n")

In [18]:
test_all_prompts()

QUESTION 1: Who is the highest paid employee?

---- Prompt V1 ----
To answer the question "Who is the highest paid employee?", the necessary table is:

- salary

---- Prompt V2 ----
{"tables": ["employees", "salary"]}

---- Prompt V3 ----
{"tables": ["salary"]}

---- Prompt V4 ----
{"tables": ["employees", "salary"]}



QUESTION 2: List employees and their field of study.

---- Prompt V1 ----
To answer the question "List employees and their field of study", the necessary table is:

1. employees
2. studies

---- Prompt V2 ----
{"tables": ["employees", "studies"]}

---- Prompt V3 ----
{"tables": ["employees", "studies"]}

---- Prompt V4 ----
{"tables": ["employees", "studies"]}



QUESTION 3: What is the average salary in 2022?

---- Prompt V1 ----
To answer the question "What is the average salary in 2022?", the necessary table is:

- salary

---- Prompt V2 ----
{"tables": ["salary"]}

---- Prompt V3 ----
{"tables": ["salary"]}

---- Prompt V4 ----
{"tables": ["salary"], "conditions": "

GPT-3.5 can correctly identify the necessary tables when provided with clear table names and short definitions.

Performance is highly sensitive to prompt structure.

Basic prompts tend to produce more verbose and less structured outputs.

Without explicit constraints, the model may hallucinate non-existent tables.

The model tends to over-select related tables even when they are not strictly required.

Enforcing a strict JSON output format reduces structural errors but does not eliminate them entirely.

Explicitly instructing the model to “use only the provided tables” significantly reduces hallucinations.

Adding a “minimum necessary tables” constraint improves precision and reduces over-selection.

Few-shot prompting produces the most stable and consistent results.

Using temperature=0 ensures reproducibility and is appropriate for controlled evaluation.